# 💳 Credit Card Fraud Detection — Analysis Notebook
Interactive version of the full pipeline. Run cells top to bottom.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print('Libraries loaded ✅')

## 1. Load Data

In [ ]:
from generate_data import generate_transactions

df = generate_transactions(n_samples=10000)
print(f'Shape: {df.shape}')
df.head(10)

## 2. Class Imbalance

In [ ]:
print(df['Class'].value_counts())
print(f"\nFraud rate: {df['Class'].mean()*100:.2f}%")

fig, ax = plt.subplots(figsize=(6,4))
df['Class'].value_counts().plot(kind='bar', ax=ax, color=['#2ecc71','#e74c3c'], edgecolor='black')
ax.set_xticklabels(['Legit', 'Fraud'], rotation=0)
ax.set_title('Class Distribution')
plt.tight_layout()
plt.show()

## 3. EDA — Amount & Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for cls, label, color in [(0,'Legit','#2ecc71'), (1,'Fraud','#e74c3c')]:
    subset = df[df['Class']==cls]
    axes[0].hist(subset['Amount'], bins=60, alpha=0.6, label=label, color=color)
    axes[1].hist(subset['Time'],   bins=60, alpha=0.6, label=label, color=color)

axes[0].set_title('Amount Distribution'); axes[0].legend()
axes[1].set_title('Time Distribution');   axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Preprocessing + SMOTE

In [ ]:
# Save data first
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/creditcard_synthetic.csv', index=False)

from preprocess import full_pipeline
X_train, X_test, y_train, y_test = full_pipeline('../data/creditcard_synthetic.csv')

print(f'X_train: {X_train.shape} | y_train fraud: {y_train.sum()}')
print(f'X_test:  {X_test.shape}  | y_test fraud:  {y_test.sum()}')

## 5. Train & Evaluate All Models

In [ ]:
from train import train_all, pick_best, plot_confusion_matrix, plot_roc_curves, plot_model_comparison

results   = train_all(X_train, y_train, X_test, y_test)
best_name = pick_best(results)

In [ ]:
from train import print_classification_report
print_classification_report(y_test, results[best_name]['y_pred'], best_name)

In [ ]:
plot_confusion_matrix(y_test, results[best_name]['y_pred'], best_name)
plot_roc_curves(results, y_test)
plot_model_comparison(results)

## 6. Real-Time Simulation

In [ ]:
from train import save_model
from predict import load_model, load_scaler, simulate_realtime

save_model(results[best_name]['model'], best_name)
model  = load_model(best_name.replace(' ', '_').lower())
scaler = load_scaler()

alerts = simulate_realtime(model, scaler, n_transactions=15, delay=0.0)